In [1]:
"""
Hands-On Quantum Computing Exercises (Qiskit)
==============================================
Requires: qiskit, qiskit-aer   ->   pip install qiskit qiskit-aer

Run the whole file:  python quantum_exercises.py
Each exercise is a self-contained function; run() at the bottom executes them
all in order and prints verification results.
"""

import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, partial_trace, entropy

SIM = AerSimulator()


# ---------------------------------------------------------------------------
# EASY: CNOT gate, control = |1>
# ---------------------------------------------------------------------------
def exercise_easy_cnot():
    print("\n=== EASY: CNOT with control qubit in |1> ===")
    qc = QuantumCircuit(2, 2)
    qc.x(0)            # prepare control (q0) in |1>
    qc.cx(0, 1)         # CNOT: q0 controls, q1 is target
    qc.measure([0, 1], [0, 1])

    print(qc.draw(output="text"))

    counts = SIM.run(qc, shots=100).result().get_counts()
    print("Counts:", counts)

    assert set(counts.keys()) == {"11"}, "Target did not flip as expected!"
    print("VERIFIED: control=1 forced target from |0> -> |1>. Only '11' observed.")


# ---------------------------------------------------------------------------
# MEDIUM: Bell state, 1024 shots
# ---------------------------------------------------------------------------
def exercise_medium_bell():
    print("\n=== MEDIUM: Bell state (|00> + |11>)/sqrt(2), 1024 shots ===")
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])

    print(qc.draw(output="text"))

    counts = SIM.run(qc, shots=1024).result().get_counts()
    print("Counts:", counts)

    assert set(counts.keys()) <= {"00", "11"}, "Unexpected outcome outside {00, 11}!"
    total = sum(counts.values())
    print(f"VERIFIED: only '00' and '01'/'11' checked -> only 00/11 appear "
          f"({counts.get('00', 0)}/{total} vs {counts.get('11', 0)}/{total}), "
          f"roughly 50/50 as expected for maximal entanglement.")


# ---------------------------------------------------------------------------
# HARD: 3-qubit GHZ state
# ---------------------------------------------------------------------------
def exercise_hard_ghz():
    print("\n=== HARD: 3-qubit GHZ state (|000> + |111>)/sqrt(2) ===")
    qc = QuantumCircuit(3, 3)
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(0, 2)
    qc.measure([0, 1, 2], [0, 1, 2])

    print(qc.draw(output="text"))

    counts = SIM.run(qc, shots=1024).result().get_counts()
    print("Counts:", counts)

    assert set(counts.keys()) <= {"000", "111"}, "Correlation broken outside {000, 111}!"
    print("VERIFIED: only '000' and '111' observed -> all three qubits are "
          "perfectly correlated (measuring any one instantly fixes the other two).")


# ---------------------------------------------------------------------------
# REAL-WORLD: Classical full adder via Toffoli + CNOT
# ---------------------------------------------------------------------------
def full_adder_circuit(a, b, cin):
    """
    q0=A, q1=B, q2=Cin  (inputs, left untouched -> reversible)
    q3=Sum (ancilla, starts |0>), q4=Cout (ancilla, starts |0>)

    Sum  = A xor B xor Cin                -> three CNOTs into q3
    Cout = AB xor A*Cin xor B*Cin (majority) -> three Toffolis into q4
    """
    qc = QuantumCircuit(5, 2)
    if a:   qc.x(0)
    if b:   qc.x(1)
    if cin: qc.x(2)

    # Sum bit
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)

    # Carry-out bit (majority function realized as XOR of pairwise ANDs)
    qc.ccx(0, 1, 4)
    qc.ccx(0, 2, 4)
    qc.ccx(1, 2, 4)

    qc.measure(3, 0)  # Sum
    qc.measure(4, 1)  # Cout
    return qc


def exercise_real_world_full_adder():
    print("\n=== REAL-WORLD: Full adder from Toffoli + CNOT gates ===")
    print(full_adder_circuit(1, 0, 1).draw(output="text"))

    print(f"{'A':>2} {'B':>2} {'Cin':>3} | {'Sum(q)':>6} {'Cout(q)':>7} | "
          f"{'Sum(c)':>6} {'Cout(c)':>7} | match")
    all_ok = True
    for a in (0, 1):
        for b in (0, 1):
            for cin in (0, 1):
                qc = full_adder_circuit(a, b, cin)
                counts = SIM.run(qc, shots=1).result().get_counts()
                bitstring = list(counts.keys())[0]     # qiskit orders c1c0 -> "cout sum"
                sum_q, cout_q = int(bitstring[-1]), int(bitstring[-2])

                total = a + b + cin
                sum_c, cout_c = total % 2, total // 2

                ok = (sum_q == sum_c) and (cout_q == cout_c)
                all_ok &= ok
                print(f"{a:>2} {b:>2} {cin:>3} | {sum_q:>6} {cout_q:>7} | "
                      f"{sum_c:>6} {cout_c:>7} | {'OK' if ok else 'MISMATCH'}")

    assert all_ok, "Quantum full adder disagreed with classical truth table!"
    print("VERIFIED: quantum circuit reproduces the classical full-adder "
          "truth table for all 8 input combinations.")


# ---------------------------------------------------------------------------
# CHALLENGE: Arbitrary (non-Bell) 2-qubit entangled state + tomography
# ---------------------------------------------------------------------------
def exercise_challenge_arbitrary_entangled(p0=0.7, p1=0.3):
    """
    Build  sqrt(p0)|01> + sqrt(p1)|10>   (unequal-amplitude entangled state,
    not one of the four standard Bell states) and verify it via the
    statevector (a simple, exact form of state tomography in simulation).
    """
    assert abs(p0 + p1 - 1.0) < 1e-9, "Probabilities must sum to 1"
    print(f"\n=== CHALLENGE: arbitrary entangled state "
          f"sqrt({p0})|01> + sqrt({p1})|10> ===")

    theta = 2 * np.arccos(np.sqrt(p0))
    qc = QuantumCircuit(2)
    qc.x(1)           # start at |01>
    qc.ry(theta, 0)   # q0: sqrt(p0)|0> + sqrt(p1)|1>  (still tensored with q1=1)
    qc.cx(0, 1)        # entangle: flips q1 when q0=1 -> sqrt(p0)|01> + sqrt(p1)|10>

    print(qc.draw(output="text"))

    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities_dict()
    print("Statevector:", sv)
    print("Measured (exact) probabilities:", probs)

    # Qiskit's little-endian bit ordering (qubit 0 = rightmost char) swaps
    # which label carries which amplitude versus the "q0,q1" way we built it.
    expected = {"01": p1, "10": p0}
    for state, p in expected.items():
        got = probs.get(state, 0.0)
        assert abs(got - p) < 1e-6, f"Amplitude mismatch on |{state}>"
    for state in probs:
        assert state in expected or probs[state] < 1e-9, f"Unexpected population in |{state}>"

    # Confirm entanglement: reduced state of qubit 0 must be mixed (not pure)
    rho0 = partial_trace(sv, [1])
    s = entropy(rho0, base=2)
    print("Reduced density matrix of qubit 0:\n", np.round(rho0.data, 3))
    print(f"Von Neumann entropy of subsystem: {s:.4f} (0 => product state, >0 => entangled)")

    assert s > 0.01, "Subsystem entropy ~0 -> state is NOT entangled!"
    print("VERIFIED: amplitudes match the target state exactly, and nonzero "
          "subsystem entropy confirms genuine entanglement (not just correlation).")


# ---------------------------------------------------------------------------
if __name__ == "__main__":
    exercise_easy_cnot()
    exercise_medium_bell()
    exercise_hard_ghz()
    exercise_real_world_full_adder()
    exercise_challenge_arbitrary_entangled()

    print("\n" + "=" * 70)
    print("REFLECTION")
    print("=" * 70)
    print("""
1. How does entanglement differ from classical correlation?

   Classical correlation (e.g. two coins secretly set to match before being
   separated) reflects a pre-existing, fixed joint state: each coin already
   "knows" its own outcome and the correlation just describes shared but
   locally determined information (a "hidden variable"). Measuring one coin
   reveals a fact that was true all along.

   Entanglement is different: before measurement, neither qubit has a
   definite individual state at all -- only the joint system does (the
   reduced state of each qubit is mixed, as the nonzero entropy in the
   Challenge exercise shows). The correlation isn't from shared prior
   information; it's built into the joint wavefunction itself. This shows up
   experimentally as violations of Bell inequalities: entangled correlations
   can be stronger than ANY classical (local hidden-variable) correlation can
   produce. Also, entangled qubits can be correlated in *multiple*,
   incompatible measurement bases simultaneously (e.g. both Z-basis and
   X-basis outcomes are correlated for a Bell pair), which has no classical
   analogue.

2. What practical difficulties did you face in scaling the circuit to 3 qubits?

   Typical friction points when moving from 2 to 3+ qubits:
   - State-space growth: the statevector size doubles per qubit (2^n), so
     inspecting/debugging amplitudes by hand quickly becomes unwieldy.
   - Bit-ordering confusion: Qiskit reports classical bit strings in
     little-endian order (rightmost = qubit 0), which is easy to
     misinterpret once there are 3+ classical bits (seen directly in the
     full-adder exercise above, where Sum and Cout had to be read from
     specific string positions).
   - Gate connectivity/depth: an extra CNOT (q0->q2) is needed to fully
     entangle the third qubit, and on real hardware (vs. this ideal
     simulator) each additional two-qubit gate adds noise, so GHZ-state
     fidelity drops faster than Bell-state fidelity as qubit count grows.
   - Ancilla management: the full-adder needed two *extra* ancilla qubits
     beyond the 3 logical inputs, and tracking which physical qubit holds
     which logical value (A, B, Cin, Sum, Cout) becomes error-prone without
     careful bookkeeping/comments.
""")


=== EASY: CNOT with control qubit in |1> ===
     ┌───┐     ┌─┐   
q_0: ┤ X ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
Counts: {'11': 100}
VERIFIED: control=1 forced target from |0> -> |1>. Only '11' observed.

=== MEDIUM: Bell state (|00> + |11>)/sqrt(2), 1024 shots ===
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
Counts: {'00': 534, '11': 490}
VERIFIED: only '00' and '01'/'11' checked -> only 00/11 appear (534/1024 vs 490/1024), roughly 50/50 as expected for maximal entanglement.

=== HARD: 3-qubit GHZ state (|000> + |111>)/sqrt(2) ===
     ┌───┐             ┌─┐   
q_0: ┤ H ├──■────■─────┤M├───
     └───┘┌─┴─┐  │  ┌─┐└╥┘   
q_1: ─────┤ X ├──┼──┤M├─╫────
          └───┘┌─┴─┐└╥┘ ║ ┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     1  0 